# ECON417 — Project 3: Industry Data Science
## Predicting Bike-Sharing Demand in Washington, D.C.

**Domain:** Industry Data Science
**Dataset:** UCI Bike Sharing Dataset — Capital Bikeshare (2011–2012)
**Research Question:** What factors drive daily bike-sharing demand,
and how accurately can we predict it?

---

### Learning Objectives
1. Work through a full demand-forecasting pipeline — the most common DS task in industry
2. Apply feature engineering to operational and weather data
3. Compare linear and nonlinear models on a real transportation dataset
4. Interpret feature importance for business decision-making

### Background

**Demand forecasting** is one of the most commercially important DS tasks:
Uber forecasts ride demand, Amazon forecasts product orders, airlines forecast
seat demand — and bike-share operators forecast daily rentals for **vehicle
rebalancing and staffing**.

This dataset covers **Capital Bikeshare** — Washington D.C.'s public bike-share
system operated by Lyft — and was donated to the UCI ML Repository.

**Why this matters in industry:**
- Underestimating demand → empty docks, lost customers
- Overestimating demand → idle bikes, wasted rebalancing costs
- A 10% RMSE reduction in demand prediction can save six figures annually
  for a mid-sized bike-share network

| Feature | Description |
|---------|-------------|
| `season` | 1=Spring, 2=Summer, 3=Fall, 4=Winter |
| `yr` | Year (0=2011, 1=2012) |
| `mnth` | Month (1–12) |
| `holiday` | Whether day is a U.S. federal holiday |
| `weekday` | Day of week (0=Sunday) |
| `workingday` | 1 if neither holiday nor weekend |
| `weathersit` | 1=Clear, 2=Cloudy, 3=Light rain/snow, 4=Heavy rain |
| `temp` | Normalized temperature (Celsius, scaled to [0,1]) |
| `atemp` | Normalized "feels like" temperature |
| `hum` | Normalized humidity |
| `windspeed` | Normalized wind speed |
| **`cnt`** | **Total daily rentals (target)** |

**Data source:** UCI ML Repository — https://archive.ics.uci.edu/ml/datasets/Bike+Sharing+Dataset


## Section 0 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_theme(style="whitegrid")
print("All libraries loaded successfully!")

## Section 1 — Data Acquisition

**Download the UCI Bike Sharing Dataset:**
```python
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip"
```

In practice, a DS analyst at a transportation company would pull this from a data
warehouse (BigQuery, Redshift, Snowflake) via SQL — but the modeling pipeline
is identical regardless of data source.

In [ ]:
import requests, zipfile
from io import BytesIO

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip"

try:
    print("Downloading UCI Bike Sharing Dataset...")
    import ssl
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode    = ssl.CERT_NONE
    resp = requests.get(url, timeout=30, verify=False)
    resp.raise_for_status()
    with zipfile.ZipFile(BytesIO(resp.content)) as z:
        df = pd.read_csv(z.open('day.csv'))
    print(f"Downloaded {len(df)} days of data  ({df['dteday'].min()} to {df['dteday'].max()})")
except Exception as e:
    print(f"Download failed ({e}). Generating simulated Capital Bikeshare data...")
    np.random.seed(42)
    n = 731  # Jan 2011 – Dec 2012 = 731 days
    season     = np.tile([1,1,1,2,2,2,3,3,3,4,4,4], 61)[:n]
    yr         = np.repeat([0, 1], [365, 366])[:n]
    mnth       = np.array([(i % 12) + 1 for i in range(n)])
    holiday    = np.random.choice([0,1], n, p=[0.97, 0.03])
    weekday    = np.array([i % 7 for i in range(n)])
    workingday = ((weekday >= 1) & (weekday <= 5) & (holiday == 0)).astype(int)
    weathersit = np.random.choice([1,2,3,4], n, p=[0.47, 0.33, 0.19, 0.01])
    temp       = np.clip(0.3 + 0.35*np.sin(np.linspace(0,4*np.pi,n)) +
                         np.random.normal(0,0.07,n), 0.05, 0.98)
    atemp      = np.clip(temp + np.random.normal(0,0.04,n), 0.05, 0.98)
    hum        = np.clip(0.6 + np.random.normal(0,0.15,n), 0.1, 0.99)
    windspeed  = np.clip(np.random.exponential(0.18, n), 0, 0.75)
    cnt_raw    = (2200 + 2800*temp - 600*(weathersit-1) + 900*workingday
                  - 700*hum - 500*windspeed + 1200*yr
                  + np.random.normal(0, 350, n))
    cnt        = np.clip(cnt_raw, 50, 8700).astype(int)
    df = pd.DataFrame({
        'instant': range(1,n+1),
        'dteday': pd.date_range('2011-01-01', periods=n).astype(str),
        'season': season, 'yr': yr, 'mnth': mnth,
        'holiday': holiday, 'weekday': weekday, 'workingday': workingday,
        'weathersit': weathersit, 'temp': temp, 'atemp': atemp,
        'hum': hum, 'windspeed': windspeed,
        'casual': (cnt*0.3).astype(int),
        'registered': (cnt*0.7).astype(int), 'cnt': cnt
    })

df.head()

## Section 2 — Feature Engineering

We select operational and weather features and add one engineered feature:
**temperature gap** (`atemp − temp`) — the difference between "feels like" and
actual temperature captures wind chill and humidity effects beyond each variable alone.

In [ ]:
feature_cols = ['season', 'yr', 'mnth', 'holiday', 'weekday',
                'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']

# Engineered feature: thermal comfort gap
df['temp_gap'] = df['atemp'] - df['temp']
feature_cols.append('temp_gap')

X = df[feature_cols].copy()
y = df['cnt'].copy()

print(f"Samples  : {len(df)} days")
print(f"Features : {len(feature_cols)}")
print(f"\nTarget (daily rentals):")
print(f"  Min    = {y.min():,}   Max  = {y.max():,}")
print(f"  Mean   = {y.mean():.0f}   Std  = {y.std():.0f}")
X.describe().round(3)

## Section 3 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(y, bins=40, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].axvline(y.mean(), color='red', ls='--', lw=2, label=f'Mean = {y.mean():.0f}')
axes[0].set_xlabel('Daily Bike Rentals')
axes[0].set_ylabel('Days')
axes[0].set_title('Distribution of Daily Bike Demand
Capital Bikeshare, Washington D.C.')
axes[0].legend()

season_labels = ['Spring', 'Summer', 'Fall', 'Winter']
season_data   = [y[df['season'] == s].values for s in [1,2,3,4]]
bp = axes[1].boxplot(season_data, labels=season_labels, patch_artist=True,
                     medianprops=dict(color='red', lw=2))
for patch, color in zip(bp['boxes'], ['#81D4FA','#FFB74D','#A5D6A7','#B0BEC5']):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[1].set_ylabel('Daily Rentals')
axes[1].set_title('Daily Demand by Season')

plt.tight_layout()
plt.show()

In [ ]:
# Continuous features vs demand
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

cont_feats = ['temp', 'atemp', 'hum', 'windspeed']
for i, feat in enumerate(cont_feats):
    axes[i].scatter(X[feat], y, alpha=0.35, s=15, color='#2196F3', edgecolors='none')
    z = np.polyfit(X[feat], y, 1)
    x_line = np.linspace(X[feat].min(), X[feat].max(), 100)
    axes[i].plot(x_line, np.poly1d(z)(x_line), 'r-', lw=2.5)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Daily Rentals')
    axes[i].set_title(f'{feat} vs Demand')

cat_feats = [('weathersit','Weather Situation'), ('weekday','Day of Week'),
             ('workingday','Working Day'),       ('mnth','Month')]
for j, (feat, label) in enumerate(cat_feats):
    groups = [y[X[feat]==v].values for v in sorted(X[feat].unique())]
    axes[j+4].boxplot(groups, labels=[str(v) for v in sorted(X[feat].unique())],
                      medianprops=dict(color='red', lw=1.5))
    axes[j+4].set_xlabel(feat)
    axes[j+4].set_ylabel('Daily Rentals' if j==0 else '')
    axes[j+4].set_title(f'{label} vs Demand')

plt.suptitle('Feature Relationships with Daily Bike Demand', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
data_full = X.copy()
data_full['cnt'] = y
corr = data_full.corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Bike Sharing — Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

print("Correlation with daily rentals (sorted by absolute value):")
corrs = corr['cnt'].drop('cnt').abs().sort_values(ascending=False)
for feat, val in corrs.items():
    bar = '█' * int(val * 30)
    print(f"  {feat:15s}: {val:.3f}  {bar}")

## Section 4 — Demeaning

After demeaning, the OLS intercept becomes the **average daily rental count**.
Each coefficient then represents the change in rentals associated with
a one-unit *above-average* change in that feature — which is exactly the
framing a product analyst would use when reporting to operations leadership.

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Compute X_mean = X.mean() and X_demeaned = X - X_mean. Print X_demeaned.mean() to verify all values are approximately zero.
raise NotImplementedError()

## Section 5 — Train / Test Split

In [ ]:
# Note: for a real time-series problem, use a chronological split.
# We use random split here for pedagogical consistency across all three projects.
X_train, X_test, y_train, y_test = train_test_split(
    X_demeaned, y, test_size=0.2, random_state=42
)
print(f"Training: {X_train.shape[0]} days  |  Test: {X_test.shape[0]} days")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

results_all = []
print("Data ready.")

## Section 6 — Evaluation Function

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Define evaluate_model(y_true, y_pred, model_name) that computes and returns R2, RMSE, and MAE using r2_score(), mean_squared_error(), mean_absolute_error()
raise NotImplementedError()

## Section 7 — OLS Regression (Baseline)

In industry, OLS is always the **first model you run** — it establishes
a performance baseline that any more complex model must meaningfully beat
to justify the added complexity and reduced interpretability.

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Fit LinearRegression() on (X_train, y_train), predict on X_test, call evaluate_model(), store result in lr_res.
raise NotImplementedError()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred_lr, alpha=0.5, s=30, color='#2196F3',
                edgecolors='white', linewidths=0.3)
lims = [0, max(y_test.max(), y_pred_lr.max()) * 1.05]
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect fit')
axes[0].set_xlabel('Actual Daily Rentals')
axes[0].set_ylabel('Predicted Daily Rentals')
axes[0].set_title('OLS — Predicted vs Actual')
axes[0].legend()

c_colors = ['#2196F3' if c > 0 else '#F44336' for c in lr.coef_]
axes[1].barh(feature_cols, lr.coef_, color=c_colors, edgecolor='white', alpha=0.87)
axes[1].axvline(0, color='black', lw=1)
axes[1].set_xlabel('Coefficient (additional daily rentals per unit feature change)')
axes[1].set_title('OLS — Regression Coefficients')

plt.tight_layout()
plt.show()

## Section 8 — Ridge Regression (L2 Regularization)

`temp` and `atemp` have a correlation > 0.99 — near-perfect multicollinearity
that inflates OLS coefficient variance. Ridge stabilizes the estimates
by imposing an L2 penalty.

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Use RidgeCV(alphas=np.logspace(-3,3,100), cv=5) on X_train_scaled. Fit Ridge(alpha=best_alpha_ridge) and predict on X_test_scaled. Call evaluate_model(), store in ridge_res.
raise NotImplementedError()

In [ ]:
alpha_range = np.logspace(-3, 3, 60)
paths_r = np.array([
    Ridge(alpha=a).fit(X_train_scaled, y_train).coef_ for a in alpha_range
])

plt.figure(figsize=(12, 6))
for i, feat in enumerate(feature_cols):
    plt.semilogx(alpha_range, paths_r[:, i], lw=2, label=feat)
plt.axvline(best_alpha_ridge, color='black', ls='--', lw=2.5,
            label=f'Best α = {best_alpha_ridge:.3f}')
plt.xlabel('α  (log scale)')
plt.ylabel('Coefficient')
plt.title('Ridge — Coefficient Path (Bike Demand Features)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## Section 9 — Lasso Regression (L1 Regularization)

Lasso will tell us which features are *truly* driving demand vs. which are
redundant given the others. If `holiday` is zeroed out, it means holiday
status adds no predictive power beyond what `workingday` already captures.

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Use LassoCV(alphas=np.logspace(-3,3,100), cv=5, max_iter=10000) on X_train_scaled. Fit Lasso(alpha=best_alpha_lasso, max_iter=10000), predict on X_test_scaled. Call evaluate_model(), store in lasso_res. Print which features are retained/zeroed.
raise NotImplementedError()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (coef, title, color) in zip(axes, [
    (lr.coef_,    'OLS',   '#2196F3'),
    (ridge.coef_, 'Ridge', '#009688'),
    (lasso.coef_, 'Lasso', '#FF5722'),
]):
    c_colors = [color if c > 0 else '#c62828' for c in coef]
    ax.barh(feature_cols, coef, color=c_colors, edgecolor='white', alpha=0.87)
    ax.axvline(0, color='black', lw=1)
    ax.set_title(f'{title} Coefficients', fontsize=11)
    ax.set_xlabel('Coefficient')
plt.suptitle('Coefficient Comparison — Bike Demand Features', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 10 — Linear Models Comparison

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Build a DataFrame from [lr_res, ridge_res, lasso_res], set 'Model' as index, round to 4 decimal places, and print the comparison table.
raise NotImplementedError()

## Section 11 — Random Forest

Bike demand is inherently nonlinear:
- Temperature has a **concave (inverted-U) effect** — demand peaks at mild temperatures
  and drops in extreme heat or cold
- Weather × weekday **interactions** matter: rain on a workday suppresses demand less
  than rain on a weekend (fewer recreational riders)

Random Forest captures these dynamics without requiring manual feature engineering.

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Create RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=3, random_state=42, n_jobs=-1), fit on (X_train, y_train), predict on X_test, call evaluate_model(), store in rf_res.
raise NotImplementedError()

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Extract rf.feature_importances_, create importance_df with columns ['Feature', 'Importance'], sort descending by Importance.
raise NotImplementedError()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

palette_rf = sns.color_palette("Blues_r", len(importance_df))
axes[0].barh(importance_df['Feature'][::-1],
             importance_df['Importance'][::-1],
             color=palette_rf, edgecolor='white', alpha=0.9)
axes[0].set_xlabel('Feature Importance')
axes[0].set_title('Random Forest — Feature Importance
(Bike Demand Prediction)')

axes[1].scatter(y_test, y_pred_rf, alpha=0.5, s=30, color='#009688',
                edgecolors='white', linewidths=0.3)
lims = [0, max(y_test.max(), y_pred_rf.max()) * 1.05]
axes[1].plot(lims, lims, 'r--', lw=2)
axes[1].set_xlabel('Actual Daily Rentals')
axes[1].set_ylabel('Predicted Daily Rentals')
axes[1].set_title('Random Forest — Predicted vs Actual')
r2_rf = r2_score(y_test, y_pred_rf)
axes[1].text(0.05, 0.95, f'R² = {r2_rf:.4f}', transform=axes[1].transAxes,
             fontsize=12, va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Bonus: visualize the nonlinear temperature effect captured by Random Forest
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter + RF partial dependence on temperature
temp_range = np.linspace(X['temp'].min(), X['temp'].max(), 100)
X_temp_sim = np.tile(X_demeaned.mean().values, (100, 1))
X_temp_sim[:, feature_cols.index('temp')] = temp_range - X_mean['temp']

axes[0].scatter(X['temp'], y, alpha=0.3, s=15, color='#2196F3', label='Observed')
axes[0].plot(temp_range, rf.predict(X_temp_sim), 'r-', lw=3,
             label='RF partial effect')
axes[0].set_xlabel('Temperature (normalized)')
axes[0].set_ylabel('Daily Rentals')
axes[0].set_title('Nonlinear Temperature Effect
(RF captures this; OLS cannot)')
axes[0].legend()

# Average demand by month
monthly_avg = df.groupby('mnth')['cnt'].mean()
axes[1].bar(monthly_avg.index, monthly_avg.values,
            color='steelblue', alpha=0.85, edgecolor='white')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Average Daily Rentals')
axes[1].set_title('Average Daily Demand by Month')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'], rotation=30)

plt.tight_layout()
plt.show()

## Section 12 — Final Comparison and Conclusion

In [ ]:
results_final = pd.DataFrame(results_all).set_index('Model').round(4)
print("All Models — Final Summary:")
print("="*58)
print(results_final.to_string())
print("="*58)
print(f"\nBest R²  : {results_final['R2'].idxmax()}  ({results_final['R2'].max():.4f})")
print(f"Best RMSE: {results_final['RMSE'].idxmin()}  ({results_final['RMSE'].min():.0f} bikes/day)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
palette = ['#1565C0', '#00695C', '#BF360C', '#2E7D32']
for i, metric in enumerate(['R2', 'RMSE', 'MAE']):
    vals = results_final[metric]
    bars = axes[i].bar(vals.index, vals, color=palette[:len(vals)],
                       edgecolor='white', alpha=0.88)
    for bar, val in zip(bars, vals):
        fmt = f'{val:.3f}' if metric == 'R2' else f'{val:.0f}'
        axes[i].text(bar.get_x() + bar.get_width()/2.,
                     bar.get_height() + max(vals)*0.01,
                     fmt, ha='center', va='bottom', fontsize=9, fontweight='bold')
    title_map = {'R2': 'R²', 'RMSE': 'RMSE (bikes/day)', 'MAE': 'MAE (bikes/day)'}
    axes[i].set_title(title_map[metric], fontsize=12)
    axes[i].set_ylabel(title_map[metric])
    axes[i].tick_params(axis='x', rotation=20)
plt.suptitle('Data Science Project — All Models Performance Comparison',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Conclusions and Business Implications

### Model Performance Takeaways

**OLS (Baseline):** Interpretable and fast — each coefficient directly answers
"how many additional daily rentals does a unit above-average temperature generate?"
Useful for quick stakeholder communication.

**Ridge:** Handles the near-perfect correlation between `temp` and `atemp`
more gracefully than OLS. The performance improvement over OLS is typically small
when features are well-engineered.

**Lasso:** Identifies the minimal feature set. If `holiday` and `workingday`
are colinear, Lasso may zero one out — a useful finding for a simpler
production model with fewer data dependencies.

**Random Forest:** Significantly outperforms all linear models on this dataset.
The key reason is the **nonlinear and interactive** nature of demand:
- Temperature effect is concave — not linear
- Weather × weekday interactions matter
- Seasonal patterns are categorical threshold effects, not linear trends

### Key Feature Importance Findings

- **Temperature** (`temp`, `atemp`) is by far the strongest driver
- **Year** (`yr`) captures system growth — D.C. Bikeshare expanded significantly
  from 2011 to 2012
- **Season** and **month** capture slower seasonal patterns
- **Humidity** and **windspeed** have moderate negative effects

### Operational Recommendations

1. **Fleet rebalancing:** Integrate the Random Forest model with the city's
   weather API to generate T+1 demand forecasts for automated dock rebalancing
2. **Staffing:** Schedule maintenance crews on low-demand days
   (cold, rainy weekends) predicted by the model
3. **Business case:** With RMSE ≈ X bikes/day, the model saves approximately
   $[RMSE × cost per misallocated bike] in daily operational costs
4. **Model deployment:** Retrain weekly using a rolling 90-day window to
   capture shifting behavioral patterns

### Discussion Questions

> 1. This project uses a random train/test split. Why is that problematic for
>    time-series demand forecasting? How would you construct a proper temporal split?
>
> 2. RMSE and MAE both measure prediction error, but in different ways. For the
>    operations team deciding how many bikes to pre-position, which metric should
>    they care about more? Why?
>
> 3. Random Forest provides feature *importance* but not *coefficients*. How would
>    you communicate the model's logic to a non-technical city transit director
>    who needs to approve its use in production?